# Explainability Analysis

In [ ]:
import jsonfrom pathlib import Pathfrom typing import Dict, List, Optional, Tuple

## GNNModelExplainer

In [ ]:
class GNNModelExplainer:    """Unified interface for explaining GNN model predictions."""    def __init__(        self,        model: GATWithAttributions,        dataset: ProteinGraphDataset,        device: torch.device = torch.device("cpu"),    ):        """        Initialize explainer.        Args:            model: Trained GATWithAttributions model            dataset: ProteinGraphDataset instance            device: Torch device for computation        """        self.model = model        self.dataset = dataset        self.device = device        self.model.to(device)        self.model.eval()        # Initialize explainer methods        self._init_gnn_explainer()        self._init_integrated_gradients()    def _init_gnn_explainer(self) -> None:        """Initialize PyG's GNNExplainer."""        try:            self.gnn_explainer = Explainer(                model=self.model,                algorithm=GNNExplainer(epochs=50, lr=0.01),                explanation_type="model",                node_mask_type="attributes",                edge_mask_type="object",                model_config={},            )            logger.info("GNNExplainer initialized")        except Exception as e:            logger.warning(f"GNNExplainer initialization failed: {e}")            self.gnn_explainer = None    def _init_integrated_gradients(self) -> None:        """Initialize Captum's Integrated Gradients."""        if not CAPTUM_AVAILABLE:            self.integrated_gradients = None            return        try:            self.integrated_gradients = IntegratedGradients(self.model)            logger.info("Integrated Gradients initialized")        except Exception as e:            logger.warning(f"Integrated Gradients initialization failed: {e}")            self.integrated_gradients = None    def explain_sample_attention(self, sample_idx: int) -> np.ndarray:        """        Compute node attributions via attention weights.        Args:            sample_idx: Index of sample to explain        Returns:            Array of per-node importance scores (num_proteins,)        """        data = self.dataset[sample_idx].to(self.device)        with torch.no_grad():            # Forward pass through GAT layers            x = data.x            for i, gat_layer in enumerate(self.model.gat_layers):                x = gat_layer(x, data.edge_index, edge_attr=data.edge_attr)                if i < len(self.model.gat_layers) - 1:                    x = F.relu(x)            # Get node importances from attribution head            node_importances = self.model.attribution_head(x).squeeze().detach().cpu().numpy()        # Normalize to [0, 1]        if node_importances.std() > 1e-6:            node_importances = (node_importances - node_importances.min()) / (                node_importances.max() - node_importances.min() + 1e-6            )        else:            node_importances = np.ones_like(node_importances) / len(node_importances)        return node_importances    def explain_sample_integrated_gradients(self, sample_idx: int) -> np.ndarray:        """        Compute node attributions via Integrated Gradients.        Args:            sample_idx: Index of sample to explain        Returns:            Array of per-node importance scores (num_proteins,)        """        if self.integrated_gradients is None:            logger.warning("Integrated Gradients not available")            return self.explain_sample_attention(sample_idx)        data = self.dataset[sample_idx].to(self.device)        # Create baseline (zero features)        baseline = torch.zeros_like(data.x)        try:            with torch.no_grad():                # Compute attributions                attr = self.integrated_gradients.attribute(                    data.x,                    baselines=baseline,                    additional_forward_args=(data.edge_index, data.edge_attr),                )            attr_scores = attr.squeeze().abs().detach().cpu().numpy()            # Normalize            if attr_scores.std() > 1e-6:                attr_scores = (attr_scores - attr_scores.min()) / (                    attr_scores.max() - attr_scores.min() + 1e-6                )            else:                attr_scores = np.ones_like(attr_scores) / len(attr_scores)            return attr_scores        except Exception as e:            logger.warning(f"Integrated Gradients computation failed: {e}")            return self.explain_sample_attention(sample_idx)    def explain_sample(        self,        sample_idx: int,        method: str = "attention",    ) -> pd.DataFrame:        """        Generate explanation for a sample.        Args:            sample_idx: Index of sample to explain            method: Attribution method ("attention" or "integrated_gradients")        Returns:            DataFrame with protein IDs and attribution scores        """        if method == "attention":            attr_scores = self.explain_sample_attention(sample_idx)        elif method == "integrated_gradients":            attr_scores = self.explain_sample_integrated_gradients(sample_idx)        else:            raise ValueError(f"Unknown method: {method}")        return pd.DataFrame(            {                "protein": self.dataset.protein_ids,                "importance": attr_scores,            }        ).sort_values("importance", ascending=False)

## ExplainabilityAnalysis

In [ ]:
class ExplainabilityAnalysis:    """Comprehensive explainability analysis for GNN models."""    def __init__(        self,        model_path: Path,        dataset: ProteinGraphDataset,        results_dir: Path,        device: torch.device = torch.device("cpu"),    ):        """        Initialize analysis.        Args:            model_path: Path to trained model checkpoint            dataset: ProteinGraphDataset instance            results_dir: Directory for saving results            device: Torch device for computation        """        self.model_path = Path(model_path)        self.dataset = dataset        self.results_dir = Path(results_dir)        self.device = device        self.results_dir.mkdir(parents=True, exist_ok=True)        # Load model        self.model = self._load_model()        self.explainer = GNNModelExplainer(self.model, self.dataset, device)    def _load_model(self) -> GATWithAttributions:        """Load trained model from checkpoint."""        if not self.model_path.exists():            raise FileNotFoundError(f"Model not found: {self.model_path}")        checkpoint = torch.load(self.model_path, map_location=self.device)        # Reconstruct model from config        config = checkpoint.get("config", {})        model = GATWithAttributions(            in_channels=config.get("in_channels", 1),            hidden_channels=config.get("hidden_channels", 128),            out_channels=config.get("out_channels", 1),            num_heads=config.get("num_heads", 4),            num_layers=config.get("num_layers", 2),            dropout=config.get("dropout", 0.3),        )        model.load_state_dict(checkpoint["model_state"])        model.to(self.device)        model.eval()        logger.info(f"Model loaded from {self.model_path}")        return model    def compute_attributions(        self,        sample_indices: Optional[List[int]] = None,        method: str = "attention",    ) -> pd.DataFrame:        """        Compute attributions for multiple samples.        Args:            sample_indices: Indices of samples to explain (None = all)            method: Attribution method        Returns:            DataFrame with protein attributions        """        if sample_indices is None:            sample_indices = list(range(len(self.dataset)))        logger.info(f"Computing {method} attributions for {len(sample_indices)} samples")        attributions = []        for i, sample_idx in enumerate(sample_indices):            if (i + 1) % 10 == 0:                logger.info(f"  Processed {i + 1}/{len(sample_indices)} samples")            attr_df = self.explainer.explain_sample(sample_idx, method=method)            attr_df["sample_idx"] = sample_idx            attributions.append(attr_df)        # Combine all attributions        all_attributions = pd.concat(attributions, ignore_index=True)        return all_attributions    def aggregate_attributions(        self,        attributions: pd.DataFrame,    ) -> pd.DataFrame:        """        Aggregate attributions across samples.        Args:            attributions: DataFrame from compute_attributions        Returns:            DataFrame with aggregated statistics        """        logger.info("Aggregating attributions across samples")        grouped = attributions.groupby("protein")["importance"].agg([            ("mean_attr", "mean"),            ("median_attr", "median"),            ("std_attr", "std"),            ("min_attr", "min"),            ("max_attr", "max"),            ("n_samples", "count"),        ]).reset_index()        # Fill NaN std with 0        grouped["std_attr"] = grouped["std_attr"].fillna(0)        # Sort by mean attribution        grouped = grouped.sort_values("mean_attr", ascending=False)        return grouped    def bootstrap_stability_analysis(        self,        sample_indices: Optional[List[int]] = None,        n_bootstrap: int = 100,        method: str = "attention",    ) -> Dict:        """        Perform bootstrap stability analysis.        Args:            sample_indices: Indices of samples to use (None = all)            n_bootstrap: Number of bootstrap samples            method: Attribution method        Returns:            Dictionary with bootstrap results and statistics        """        if sample_indices is None:            sample_indices = list(range(len(self.dataset)))        logger.info(f"Running bootstrap stability analysis ({n_bootstrap} iterations)")        n_samples = len(sample_indices)        top_protein_frequencies = {protein: 0 for protein in self.dataset.protein_ids}        all_bootstrap_ranks = {protein: [] for protein in self.dataset.protein_ids}        for iteration in range(n_bootstrap):            if (iteration + 1) % 10 == 0:                logger.info(f"  Bootstrap iteration {iteration + 1}/{n_bootstrap}")            # Resample with replacement            bootstrap_indices = np.random.choice(sample_indices, size=n_samples, replace=True)            # Compute attributions for bootstrap sample            attr = self.compute_attributions(bootstrap_indices.tolist(), method=method)            agg = self.aggregate_attributions(attr)            # Track top-100 proteins            top_100 = agg["protein"].head(100).values            for protein in top_100:                top_protein_frequencies[protein] += 1            # Track all protein ranks            protein_rank_map = {p: i for i, p in enumerate(agg["protein"].values)}            for protein in self.dataset.protein_ids:                rank = protein_rank_map.get(protein, len(self.dataset.protein_ids))                all_bootstrap_ranks[protein].append(rank)        # Compute bootstrap statistics        bootstrap_stats = []        for protein in self.dataset.protein_ids:            ranks = all_bootstrap_ranks[protein]            bootstrap_stats.append({                "protein": protein,                "bootstrap_frequency_top100": top_protein_frequencies[protein],                "bootstrap_mean_rank": np.mean(ranks),                "bootstrap_std_rank": np.std(ranks),                "bootstrap_median_rank": np.median(ranks),            })        bootstrap_df = pd.DataFrame(bootstrap_stats).sort_values(            "bootstrap_frequency_top100", ascending=False        )        return {            "bootstrap_results": bootstrap_df,            "top_protein_frequencies": top_protein_frequencies,            "all_bootstrap_ranks": all_bootstrap_ranks,            "n_bootstrap": n_bootstrap,        }    def generate_protein_attribution_report(        self,        train_indices: Optional[List[int]] = None,        n_bootstrap: int = 100,        method: str = "attention",    ) -> pd.DataFrame:        """        Generate comprehensive protein attribution report.        Args:            train_indices: Indices of training samples to analyze            n_bootstrap: Number of bootstrap iterations            method: Attribution method        Returns:            DataFrame with complete protein attribution statistics        """        if train_indices is None:            train_indices = list(range(len(self.dataset)))        # Compute attributions        attributions = self.compute_attributions(train_indices, method=method)        agg_attrs = self.aggregate_attributions(attributions)        # Bootstrap analysis        bootstrap_results = self.bootstrap_stability_analysis(            train_indices, n_bootstrap=n_bootstrap, method=method        )        bootstrap_df = bootstrap_results["bootstrap_results"]        # Merge results        report = agg_attrs.merge(bootstrap_df[["protein", "bootstrap_frequency_top100"]], on="protein")        # Reorder columns        report = report[[            "protein",            "mean_attr",            "median_attr",            "std_attr",            "min_attr",            "max_attr",            "n_samples",            "bootstrap_frequency_top100",        ]]        return report.sort_values("mean_attr", ascending=False)    def plot_attribution_stability(        self,        all_bootstrap_ranks: Dict[str, List[int]],        top_n: int = 20,        figsize: Tuple[int, int] = (12, 6),    ) -> Figure:        """        Plot distribution of attribution stability.        Args:            all_bootstrap_ranks: Dictionary from bootstrap_stability_analysis            top_n: Number of top proteins to plot            figsize: Figure size        Returns:            Matplotlib Figure        """        # Compute stability metrics        stability_data = []        for protein, ranks in all_bootstrap_ranks.items():            stability_data.append({                "protein": protein,                "mean_rank": np.mean(ranks),                "std_rank": np.std(ranks),                "cv_rank": np.std(ranks) / (np.mean(ranks) + 1e-6),  # Coefficient of variation            })        stability_df = pd.DataFrame(stability_data).sort_values("mean_rank")        # Plot top proteins by stability (low std)        top_stable = stability_df.nsmallest(top_n, "std_rank")        fig, axes = plt.subplots(2, 2, figsize=figsize)        # 1. Mean rank distribution        axes[0, 0].hist(stability_df["mean_rank"], bins=30, edgecolor="black", alpha=0.7)        axes[0, 0].set_xlabel("Mean Bootstrap Rank")        axes[0, 0].set_ylabel("Count")        axes[0, 0].set_title("Distribution of Mean Bootstrap Rank")        axes[0, 0].grid(alpha=0.3)        # 2. Std rank distribution        axes[0, 1].hist(stability_df["std_rank"], bins=30, edgecolor="black", alpha=0.7)        axes[0, 1].set_xlabel("Std Bootstrap Rank")        axes[0, 1].set_ylabel("Count")        axes[0, 1].set_title("Distribution of Bootstrap Rank Variability")        axes[0, 1].grid(alpha=0.3)        # 3. Top stable proteins        y_pos = np.arange(len(top_stable))        axes[1, 0].barh(y_pos, top_stable["mean_rank"].values, xerr=top_stable["std_rank"].values)        axes[1, 0].set_yticks(y_pos)        axes[1, 0].set_yticklabels(top_stable["protein"].values, fontsize=8)        axes[1, 0].set_xlabel("Mean Bootstrap Rank")        axes[1, 0].set_title(f"Top {top_n} Most Stable Proteins")        axes[1, 0].invert_yaxis()        axes[1, 0].grid(alpha=0.3, axis="x")        # 4. Mean vs Std rank scatter        scatter = axes[1, 1].scatter(            stability_df["mean_rank"],            stability_df["std_rank"],            alpha=0.6,            s=50,            c=stability_df["cv_rank"],            cmap="viridis",        )        axes[1, 1].set_xlabel("Mean Bootstrap Rank")        axes[1, 1].set_ylabel("Std Bootstrap Rank")        axes[1, 1].set_title("Rank Stability (Mean vs Variability)")        axes[1, 1].grid(alpha=0.3)        plt.colorbar(scatter, ax=axes[1, 1], label="Coefficient of Variation")        plt.tight_layout()        return fig    def save_results(        self,        report: pd.DataFrame,        plot: Figure,        output_name: str = "protein_attributions",    ) -> None:        """        Save analysis results.        Args:            report: Protein attribution report DataFrame            plot: Matplotlib figure            output_name: Base name for output files        """        # Save CSV        csv_path = self.results_dir / f"{output_name}.csv"        report.to_csv(csv_path, index=False)        logger.info(f"Saved protein attributions to {csv_path}")        # Save plot        plot_path = self.results_dir / f"{output_name}_stability.png"        plot.savefig(plot_path, dpi=300, bbox_inches="tight")        logger.info(f"Saved stability plot to {plot_path}")        # Save summary JSON        summary = {            "total_proteins": len(report),            "top_10_proteins": report.head(10)["protein"].tolist(),            "top_100_proteins": report.head(100)["protein"].tolist(),            "mean_attribution_stats": {                "mean": float(report["mean_attr"].mean()),                "std": float(report["mean_attr"].std()),                "min": float(report["mean_attr"].min()),                "max": float(report["mean_attr"].max()),            },        }        json_path = self.results_dir / f"{output_name}_summary.json"        with open(json_path, "w") as f:            json.dump(summary, f, indent=2)        logger.info(f"Saved summary to {json_path}")

## explain_model

In [ ]:
def explain_model(config_path: str = "config.yaml"):    """    Generate model explanations.    Args:        config_path: Path to configuration file    """    config = load_config(config_path)    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    results = {}    for cohort in ["ROSMAP", "MSBB", "MAYO"]:        logger.info(f"\n{'='*60}")        logger.info(f"Explainability Analysis: {cohort}")        logger.info(f"{'='*60}")        try:            # Paths            processed_dir = Path(config["data"]["processed_dir"])            graphs_dir = Path("data/graphs")            checkpoint_dir = Path(config["logging"]["checkpoint_dir"])            results_dir = checkpoint_dir / "explainability"            processed_file = processed_dir / f"{cohort}_processed.h5ad"            graph_file = graphs_dir / f"{cohort}_graph.graphml"            model_file = checkpoint_dir / f"{cohort}_gnn_model.pt"            if not all(p.exists() for p in [processed_file, graph_file, model_file]):                logger.warning(f"Missing files for {cohort}, skipping")                continue            # Load dataset            dataset = ProteinGraphDataset(                graph_path=graph_file,                processed_data_path=processed_file,                seed=config.get("debug", {}).get("seed", 42),            )            # Get train indices            train_indices, _, _ = dataset.get_train_val_test_split()            # Run analysis            analyzer = ExplainabilityAnalysis(                model_path=model_file,                dataset=dataset,                results_dir=results_dir,                device=device,            )            # Generate report            report = analyzer.generate_protein_attribution_report(                train_indices=train_indices,                n_bootstrap=100,                method="attention",            )            # Bootstrap analysis for plotting            bootstrap_res = analyzer.bootstrap_stability_analysis(                sample_indices=train_indices,                n_bootstrap=100,                method="attention",            )            # Generate plot            fig = analyzer.plot_attribution_stability(                bootstrap_res["all_bootstrap_ranks"],                top_n=20,            )            # Save results            analyzer.save_results(report, fig, output_name=f"{cohort}_protein_attributions")            results[cohort] = {                "report": report,                "bootstrap_results": bootstrap_res,            }            logger.info(f"Explainability analysis complete for {cohort}")        except Exception as e:            logger.error(f"Error analyzing {cohort}: {e}")            results[cohort] = {"error": str(e)}    return results

## Main Execution

In [ ]:
if __name__ == "__main__":    import argparse    parser = argparse.ArgumentParser(description="Generate model explanations")    parser.add_argument("--config", default="config.yaml", help="Config file path")    args = parser.parse_args()    explain_model(args.config)